A function that takes merged monthly netcdf file and converts it to seasonal

Monthly netcdf file:
- Lat
- Long
- predicted precip
- actual precip
- date
- lead time

Seasonal netcdf file:
- lat
- long
- predicted precip
- actual precip
- date
- lead time category
- (month - lead time) for seasonality
- season

In [1]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd

In [2]:
# import google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# load an example merged netcdf file
merged_monthly_file = xr.open_dataset('/content/drive/MyDrive/capstone_data/netCDF/eastern_east_africa_CanESM5_merged.nc')
merged_monthly_file_df = merged_monthly_file.to_dataframe().reset_index().dropna()

In [4]:
# seasons dictionary for month minus lead time

# if month_minus_lead is [9.5, 8.5, 7.5], then those squares are all OND short lead (0-2 months)
# the rest of the seasons must be defined accordingly by manual input
# short:0-1, med: 2-3, long: 4-6 months
# procedure: take the full monthly data for a given region, keep only the months
# of the seasons for that region. make a second season column if needed
# example: southern africa seasons = DJF, FMA
# subset the data so that there are only DJF and FMA months,
# in the season 1 column assign DJF short/med/long,
# copy the dataframe, and in the season 2 column assign FMA short/med/long,
# NA if otherwise, merge those 2 dataframes to get the final dataframe
# convert to netcdf
# if there are 3 seasons, adjust accordingly
'''
[[11.5 10.5  9.5  8.5  7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5]
 [10.5  9.5  8.5  7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5]
 [ 9.5  8.5  7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5]
 [ 8.5  7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5]
 [ 7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5]
 [ 6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5]
 [ 5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5]
 [ 4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5]
 [ 3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5 -7.5]
 [ 2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5 -7.5 -8.5]
 [ 1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5 -7.5 -8.5 -9.5]
 [ 0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5 -7.5 -8.5 -9.5 -10.5]]

'''
regions_seasons_dict = {
    'eastern_east_africa': {
        'OND': {'OND_short': [9.5, 8.5],
                'OND_medium': [7.5, 6.5],
                'OND_long': [5.5, 4.5, 3.5]},
        'MAM': {'MAM_short': [2.5, 1.5],
                'MAM_medium': [0.5, -0.5],
                'MAM_long': [-1.5, -2.5, -3.5]}
    },
    'lake_victoria': {
        'DJF': {'DJF_short': [11.5, 10.5, -0.5, -1.5],
                'DJF_medium': [9.5, 8.5, -2.5, -3.5],
                'DJF_long': [7.5, 6.5, 5.5, -4.5, -5.5, -6.5]},
         'MAM': {'MAM_short': [2.5, 1.5],
                'MAM_medium': [0.5, -0.5],
                'MAM_long': [-1.5, -2.5, -3.5]},
        'SON': {'SON_short': [8.5, 7.5],
                'SON_medium': [6.5, 5.5],
                'SON_long': [4.5, 3.5, 2.5]}
    },
    'west_africa': {
        'JAS': {'JAS_short': [],
                'JAS_medium': [],
                'JAS_long': []}
    },
    'southern_africa': {
        'DJF': {'DJF_short': [],
                'DJF_medium': [],
                'DJF_long': []},
        'FMA': {'FMA_short': [],
                'FMA_medium': [],
                'FMA_long': []}
    },
    'south_sudan': {
        'MJJ': {'MJJ_short': [],
                'MJJ_medium': [],
                'MJJ_long': []},
        'JAS': {'JAS_short': [],
                'JAS_medium': [],
                'JAS_long': []},
        'ASO': {'ASO_short': [],
                'ASO_medium': [],
                'ASO_long': []}
    },
    'eastern_ukraine': {
        'DJF': {'DJF_short': [],
                'DJF_medium': [],
                'DJF_long': []},
        'AMJ': {'AMJ_short': [],
                'AMJ_medium': [],
                'AMJ_long': []},
        'JA': {'JA_short': [],
               'JA_medium': [],
               'JA_long': []}
    },
    'sri_lanka': {
        'OND': {'OND_short': [],
                'OND_medium': [],
                'OND_long': []}
    }
}



In [5]:
# seperate month and year into seperate columns
merged_monthly_file_df['month'] = merged_monthly_file_df['time'].dt.month
merged_monthly_file_df['year'] = merged_monthly_file_df['time'].dt.year

merged_monthly_file_df['month_minus_lead_time'] = merged_monthly_file_df['month'] - merged_monthly_file_df['lead_time']

merged_monthly_file_df

,lead_time,time,M,latitude,longitude,predicted_precip,precip,month,year,month_minus_lead_time
0,0.5,1991-01-01,1.0,-3.5,38.0,3.096710,18.434658,1,1991,0.5
1,0.5,1991-01-01,1.0,-3.5,38.5,3.096710,21.333597,1,1991,0.5
2,0.5,1991-01-01,1.0,-3.5,39.0,1.101388,17.430561,1,1991,0.5
3,0.5,1991-01-01,1.0,-3.5,39.5,1.101388,11.300217,1,1991,0.5
25,0.5,1991-01-01,1.0,-3.0,38.0,3.144489,28.934315,1,1991,0.5
...,...,...,...,...,...,...,...,...,...,...
54215994,11.5,2021-11-01,20.0,8.0,47.5,0.675175,1.443916,11,2021,-0.5
54215995,11.5,2021-11-01,20.0,8.0,48.0,0.444444,2.984410,11,2021,-0.5
54215996,11.5,2021-11-01,20.0,8.0,48.5,0.444444,2.092375,11,2021,-0.5
54215997,11.5,2021-11-01,20.0,8.0,49.0,0.875061,3.401675,11,2021,-0.5
